In [ ]:
import numpy as np
import pandas as pd
from data import (
    Vertebral_column,
    Vertebral_column_KF,
    Ecoli_TestSize,
    Jm1_TestSize,
    Jm1_SMOTETomek,
    Haberman_TestSize,
    Transfution_TestSize,
    Pima_TestSize,
    Co_Author_TestSize,
    churn,
    Churn_SMOTETomek_IR,
    Abanole_TestSize,
    Abanole_SMOTETomek,
    Ecoli_SMOTETomek_IR,
    Haberman_SMOTETomek_IR,
    Pima_SMOTETomek_IR,
    Transfution_SMOTETomek_IR,
    Co_Author_SMOTETomek_IR,
    Yeast_TestSize,
    Yeast_SMOTETomek,
    Vertebral_column_SMOTETomek_IR,
)
import trainning_of_adaboost as toa
from sklearn.ensemble import AdaBoostClassifier
import adaboost_svm, ImAda_DecisionTree
from report import report
from sklearn.metrics import classification_report, precision_recall_fscore_support as score
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, f1_score, precision_score
import math
from datetime import datetime
import csv
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from wsvm.application import Wsvm
from sklearn.svm import SVC

In [ ]:
def compute_metrics(y_test,y_pred):
    cm_WSVM = confusion_matrix(y_test, y_pred)
    se = cm_WSVM[1,1]/(cm_WSVM[1,0]+cm_WSVM[1,1])
    sp = cm_WSVM[0,0]/(cm_WSVM[0,0]+cm_WSVM[0,1])
    gmean = math.sqrt(se*sp)
    f1s = f1_score(y_test,y_pred)
    acc = accuracy_score(y_test,y_pred)
    pre = precision_score(y_test,y_pred)
    auc = roc_auc_score(y_test, y_pred)

    return sp, se, gmean, f1s, pre, acc, auc, cm_WSVM

In [ ]:
# 1. SVM lib
def svm_lib(X_train, y_train,X_test):
    clf = SVC(probability=True, kernel='linear')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return y_pred

In [ ]:
# 3. DecisionTree
from sklearn import tree
def decisiontree(X_train, y_train,X_test):
    clf = tree.DecisionTreeClassifier()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return y_pred

In [ ]:
# 4. WSVM
def wsvm(C,X_train, y_train,X_test,distribution_weight=None):
    model = Wsvm(C,distribution_weight)
    model.fit(X_train, y_train)
    test_pred = model.predict(X_test)
    return test_pred

In [ ]:
# 5. AdaBoost SVM
def ada_svm(X_train, y_train, X_test):
    clf = AdaBoostClassifier(SVC(probability=True,kernel='linear'),n_estimators=100,learning_rate=1.0, algorithm='SAMME')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return y_pred


In [ ]:
# 6. AdaBoost DecisionTree
def ada_decisiontree(X_train, y_train,X_test):
    clf = AdaBoostClassifier(n_estimators=100)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return y_pred

In [ ]:
# 7. AdaBoost WSVM
def ada_wsvm(M, C, theta, X_train, y_train, X_test):
    w, b, a = toa.fit(X_train, y_train, M, C, instance_categorization=True, proposed_preprocessing=False, proposed_alpha=False, test_something=False, theta=theta)
    y_pred = toa.predict(X_test, w, b, a, M)
    return y_pred, a

In [ ]:
#10. IM.AdaBoost-12 WSVM
def imada_12_wsvm(M, C, theta, X_train, y_train, X_test):
    w, b, a = toa.fit(X_train, y_train, M, C, instance_categorization=True, proposed_preprocessing=True, proposed_alpha=True, test_something=False, theta=theta)
    y_pred = toa.predict(X_test, w, b, a, M)
    return y_pred, a

In [ ]:
#13. IM.AdaBoost12 + SVM
def imada_12_svm(M, C, theta, X_train, y_train, X_test):
    w, b, a = toa.fit(X_train, y_train, M, C, instance_categorization=False, proposed_preprocessing=True, proposed_alpha=True, test_something=False, theta=theta)
    y_pred = toa.predict(X_test, w, b, a, M)
    return y_pred, a

In [ ]:
def imada_12_decisiontree(M, theta, X_train, y_train, X_test):
    clf, a = ImAda_DecisionTree.fit(
        X_train, y_train, M, proposed_preprocessing=True, proposed_alpha=True, theta=theta
    )
    y_pred = ImAda_DecisionTree.predict(X_test, a, clf)
    return y_pred, a


# EANR-AdaBoost1: thay init bang entropy_init_weight
def eanr_adaboost1_wsvm(M, C, theta, X_train, y_train, X_test):
    w, b, a = toa.fit(
        X_train, y_train, M, C,
        instance_categorization=True,
        proposed_preprocessing=True,
        proposed_alpha=True,
        test_something=False,
        theta=theta,
        use_entropy_init=True,
        use_noise_robust_confident=False,
    )
    y_pred = toa.predict(X_test, w, b, a, M)
    return y_pred, a


def eanr_adaboost1_svm(M, C, theta, X_train, y_train, X_test):
    w, b, a = toa.fit(
        X_train, y_train, M, C,
        instance_categorization=False,
        proposed_preprocessing=True,
        proposed_alpha=True,
        test_something=False,
        theta=theta,
        use_entropy_init=True,
        use_noise_robust_confident=False,
    )
    y_pred = toa.predict(X_test, w, b, a, M)
    return y_pred, a


def eanr_adaboost1_decisiontree(M, theta, X_train, y_train, X_test):
    clf, a = ImAda_DecisionTree.fit(
        X_train, y_train, M,
        proposed_preprocessing=True,
        proposed_alpha=True,
        theta=theta,
        use_entropy_init=True,
        use_noise_robust_confident=False,
    )
    y_pred = ImAda_DecisionTree.predict(X_test, a, clf)
    return y_pred, a


# EANR-AdaBoost2: thay confident bang noise_robust_confident
def eanr_adaboost2_wsvm(M, C, theta, X_train, y_train, X_test):
    w, b, a = toa.fit(
        X_train, y_train, M, C,
        instance_categorization=True,
        proposed_preprocessing=True,
        proposed_alpha=True,
        test_something=False,
        theta=theta,
        use_entropy_init=False,
        use_noise_robust_confident=True,
    )
    y_pred = toa.predict(X_test, w, b, a, M)
    return y_pred, a


def eanr_adaboost2_svm(M, C, theta, X_train, y_train, X_test):
    w, b, a = toa.fit(
        X_train, y_train, M, C,
        instance_categorization=False,
        proposed_preprocessing=True,
        proposed_alpha=True,
        test_something=False,
        theta=theta,
        use_entropy_init=False,
        use_noise_robust_confident=True,
    )
    y_pred = toa.predict(X_test, w, b, a, M)
    return y_pred, a


def eanr_adaboost2_decisiontree(M, theta, X_train, y_train, X_test):
    clf, a = ImAda_DecisionTree.fit(
        X_train, y_train, M,
        proposed_preprocessing=True,
        proposed_alpha=True,
        theta=theta,
        use_entropy_init=False,
        use_noise_robust_confident=True,
    )
    y_pred = ImAda_DecisionTree.predict(X_test, a, clf)
    return y_pred, a


# EANR-AdaBoost: thay ca init + confident
def eanr_adaboost_wsvm(M, C, theta, X_train, y_train, X_test):
    w, b, a = toa.fit(
        X_train, y_train, M, C,
        instance_categorization=True,
        proposed_preprocessing=True,
        proposed_alpha=True,
        test_something=False,
        theta=theta,
        use_entropy_init=True,
        use_noise_robust_confident=True,
    )
    y_pred = toa.predict(X_test, w, b, a, M)
    return y_pred, a


def eanr_adaboost_svm(M, C, theta, X_train, y_train, X_test):
    w, b, a = toa.fit(
        X_train, y_train, M, C,
        instance_categorization=False,
        proposed_preprocessing=True,
        proposed_alpha=True,
        test_something=False,
        theta=theta,
        use_entropy_init=True,
        use_noise_robust_confident=True,
    )
    y_pred = toa.predict(X_test, w, b, a, M)
    return y_pred, a


def eanr_adaboost_decisiontree(M, theta, X_train, y_train, X_test):
    clf, a = ImAda_DecisionTree.fit(
        X_train, y_train, M,
        proposed_preprocessing=True,
        proposed_alpha=True,
        theta=theta,
        use_entropy_init=True,
        use_noise_robust_confident=True,
    )
    y_pred = ImAda_DecisionTree.predict(X_test, a, clf)
    return y_pred, a

In [ ]:
####################################### TEST SIZE SCRIPT - FIND BEST PARAMETERS ################################
# M=[10,20,30,40,50]
# C=[0.1,10,100,1000,10000]
# theta = [0.5,1,1.5,2,2.5]
# M=[10,15,20,25]
# C=[10,100,1000,5000,10000]
# theta = [0.3, 0.5, 0.7, 1,1.5,2]
# N = 1
# test_size = [0.2]
# dataset = Co_Author_TestSize

# time = datetime.now().strftime("%d%m%Y_%H%M%S")
# filename = (str(dataset).split("\\")[-1]).split(".")[0]
# filepath = f'./Experiment/Data_{filename}_TestSize.csv'
# for n in range(0, N):
#     header = ['Test Size','Method', 'M','C','theta','SP', 'SE', 'Gmean', 'F1 Score','Precision','Accuracy','AUC','Ma tran nham lan','List of err_w','List of alpha']
#     data = []
#     print("Lan boc: ", n + 1)
#     for testsize in test_size:
#         X_train, y_train, X_test, y_test = dataset.load_data(test_size=testsize)
#         #No 1
#         print("Decision Tree starting...\n")
#         y_pred = decisiontree(X_train, y_train, X_test)
#         sp, se, gmean, f1s, pre, acc, auc, cm_WSVM = compute_metrics(y_test, y_pred)
#         name = "Decision Tree"
#         le = "None"
#         la = "None"
#         m = "None"
#         c = "none"
#         t = "none"
#         data.append([testsize, name, m, c, t, sp, se, gmean, f1s, pre, acc, auc, str(cm_WSVM), le, la])

#         #No 2
#         print("SVM (lib) starting...\n")
#         y_pred = svm_lib(X_train, y_train, X_test)
#         sp, se, gmean, f1s, pre, acc, auc, cm_WSVM = compute_metrics(y_test, y_pred)
#         name = "SVM (lib)"
#         le = "None"
#         la = "None"
#         m = "None"
#         c = "none"
#         t = "none"
#         data.append([testsize, name, m, c, t, sp, se, gmean, f1s, pre, acc, auc, str(cm_WSVM), le, la])

#         #No 3
#         print("ADA_Decision Tree starting...\n")
#         y_pred = ada_decisiontree(X_train, y_train, X_test)
#         sp, se, gmean, f1s, pre, acc, auc, cm_WSVM = compute_metrics(y_test, y_pred)
#         name = 'ADA_DSTree'
#         le = "None"
#         la = "None"
#         m = "None"
#         c = "none"
#         t = "none"
#         data.append([testsize, name, m, c, t, sp, se, gmean, f1s, pre, acc, auc, str(cm_WSVM), le, la])

#         #No 4
#         print("ADA_SVM starting...\n")
#         y_pred = ada_svm(X_train, y_train, X_test)
#         sp, se, gmean, f1s, pre, acc, auc, cm_WSVM = compute_metrics(y_test, y_pred)
#         name = 'ADA_SVM'
#         le = "None"
#         la = "None"
#         m = "None"
#         c = "none"
#         t = "none"
#         data.append([testsize, name, m, c, t, sp, se, gmean, f1s, pre, acc, auc, str(cm_WSVM), le, la])

#         for m in M:
#             for c in C:
#                 for t in theta:
#                     print(m, c, t)
#                     #No 5
#                     print("WSVM starting...\n")
#                     num_samples, _ = X_train.shape
#                     distribution_weight = np.ones(num_samples)
#                     y_pred = wsvm(c, X_train, y_train, X_test, distribution_weight)
#                     sp, se, gmean, f1s, pre, acc, auc, cm_WSVM = compute_metrics(y_test, y_pred)
#                     name = "WSVM"
#                     le = "None"
#                     la = "None"
#                     data.append([testsize, name, m, c, t, sp, se, gmean, f1s, pre, acc, auc, str(cm_WSVM), le, la])

#                     #No 6
#                     print("ADA_WSVM starting...\n")
#                     y_pred, a = ada_wsvm(m, c, t, X_train, y_train, X_test)
#                     sp, se, gmean, f1s, pre, acc, auc, cm_WSVM = compute_metrics(y_test, y_pred)
#                     name = 'ADA_WSVM'
#                     data.append([testsize, name, m, c, t, sp, se, gmean, f1s, pre, acc, auc, str(cm_WSVM), 'None', a])

#                     #No 9
#                     print("ImADA_12_DecisionTree starting...\n")
#                     y_pred, a = imada_12_decisiontree(m, t, X_train, y_train, X_test)
#                     sp, se, gmean, f1s, pre, acc, auc, cm_WSVM = compute_metrics(y_test, y_pred)
#                     name = 'ImADA_12_DecisionTree'
#                     data.append([testsize, name, m, c, t, sp, se, gmean, f1s, pre, acc, auc, str(cm_WSVM), 'None', a])

#                     #No 12
#                     print("ImADA_12_SVM starting...\n")
#                     y_pred, a = imada_12_svm(m, c, t, X_train, y_train, X_test)
#                     sp, se, gmean, f1s, pre, acc, auc, cm_WSVM = compute_metrics(y_test, y_pred)
#                     name = 'ImADA_12_SVM'
#                     data.append([testsize, name, m, c, t, sp, se, gmean, f1s, pre, acc, auc, str(cm_WSVM), 'None', a])

#                     #No 15
#                     print("ImADA_12_WSVM starting...\n")
#                     y_pred, a = imada_12_wsvm(m, c, t, X_train, y_train, X_test)
#                     sp, se, gmean, f1s, pre, acc, auc, cm_WSVM = compute_metrics(y_test, y_pred)
#                     name = 'ImADA_12_WSVM'
#                     data.append([testsize, name, m, c, t, sp, se, gmean, f1s, pre, acc, auc, str(cm_WSVM), 'None', a])

#     with open(f'./Experiment/Data_{filename}_{time}_TestSize.csv', 'a', encoding='UTF8', newline='') as f1:
#         writer = csv.writer(f1)
#         writer.writerow(header)
#         writer.writerows(data)

In [ ]:
####################################### TEST SIZE SCRIPT - FIND BEST PARAMETERS ################################
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import threading

# M=[10,20,30,40,50]
# C=[0.1,10,100,1000,10000]
# theta = [0.5,1,1.5,2,2.5]
M = [10, 15, 20, 25]
C = [10, 100, 1000, 5000, 10000]
theta = [0.3, 0.5, 0.7, 1, 1.5, 2]
N = 1
test_size = [0.2]

# Chon dataset goc + dataset sau can bang SMOTETomek
# Doi 2 bien duoi day theo dataset can chay.
dataset_name = "Jm1"
dataset_original = Jm1_TestSize
dataset_smote = Jm1_SMOTETomek

# Neu dataset co tham so new_rate (vd: churn, ecoli, pima, haberman, transfution, co-author, vertebral)
# dat gia tri (vd: new_rate = 0.2). Neu khong can, de None.
new_rate = None

time = datetime.now().strftime("%d%m%Y_%H%M%S")
filepath = f'./Experiment/Data_{dataset_name}_{time}_TestSize.csv'
max_workers = min(8, (os.cpu_count() or 4))

header = ['Test Size', 'Method', 'M', 'C', 'theta', 'SP', 'SE', 'Gmean', 'F1 Score', 'Precision', 'Accuracy', 'AUC', 'Ma tran nham lan', 'List of err_w', 'List of alpha']
file_lock = threading.Lock()

# Tao file output va ghi header ngay tu dau de tranh mat du lieu khi dang chay.
with open(filepath, 'a', encoding='UTF8', newline='') as f1:
    writer = csv.writer(f1)
    writer.writerow(header)


def append_row(row):
    with file_lock:
        with open(filepath, 'a', encoding='UTF8', newline='') as f1:
            writer = csv.writer(f1)
            writer.writerow(row)


def load_data_flexible(dataset_module, testsize, new_rate_val=None):
    # Ho tro ca 2 dang ham: load_data(test_size) va load_data(test_size, new_rate)
    if new_rate_val is None:
        return dataset_module.load_data(test_size=testsize)
    try:
        return dataset_module.load_data(test_size=testsize, new_rate=new_rate_val)
    except TypeError:
        return dataset_module.load_data(test_size=testsize)


def safe_run_and_append(method_name, testsize, m, c, t, y_test, run_fn):
    try:
        y_pred, alpha = run_fn()
        sp, se, gmean, f1s, pre, acc, auc, cm = compute_metrics(y_test, y_pred)
        append_row([testsize, method_name, m, c, t, sp, se, gmean, f1s, pre, acc, auc, str(cm), 'None', alpha])
    except Exception as ex:
        # Luu loi vao file de de theo doi khi chay dai.
        append_row([testsize, method_name, m, c, t, 'ERR', 'ERR', 'ERR', 'ERR', 'ERR', 'ERR', 'ERR', str(ex), 'None', 'None'])


dataset_variants = [
    ("ORIG", dataset_original),
    ("SMOTE", dataset_smote),
]

for n in range(0, N):
    print("Lan boc:", n + 1)
    for testsize in test_size:
        for variant_tag, dataset_module in dataset_variants:
            print(f"Dataset variant: {variant_tag}")
            X_train, y_train, X_test, y_test = load_data_flexible(
                dataset_module, testsize, new_rate_val=new_rate
            )
            num_samples, _ = X_train.shape
            distribution_weight = np.ones(num_samples)

            futures = []
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                # Nhom thuat toan khong dung M, C, theta
                futures.append(executor.submit(
                    safe_run_and_append,
                    f"Decision Tree | {variant_tag}", testsize, "None", "none", "none", y_test,
                    lambda: (decisiontree(X_train, y_train, X_test), "None")
                ))
                futures.append(executor.submit(
                    safe_run_and_append,
                    f"SVM (lib) | {variant_tag}", testsize, "None", "none", "none", y_test,
                    lambda: (svm_lib(X_train, y_train, X_test), "None")
                ))
                futures.append(executor.submit(
                    safe_run_and_append,
                    f"ADA_DSTree | {variant_tag}", testsize, "None", "none", "none", y_test,
                    lambda: (ada_decisiontree(X_train, y_train, X_test), "None")
                ))
                futures.append(executor.submit(
                    safe_run_and_append,
                    f"ADA_SVM | {variant_tag}", testsize, "None", "none", "none", y_test,
                    lambda: (ada_svm(X_train, y_train, X_test), "None")
                ))

                # Nhom thuat toan dung M, C, theta
                for m in M:
                    for c in C:
                        for t in theta:
                            print(variant_tag, m, c, t)
                            futures.append(executor.submit(
                                safe_run_and_append,
                                f"WSVM | {variant_tag}", testsize, m, c, t, y_test,
                                lambda m=m, c=c: (wsvm(c, X_train, y_train, X_test, distribution_weight), "None")
                            ))
                            futures.append(executor.submit(
                                safe_run_and_append,
                                f"ADA_WSVM | {variant_tag}", testsize, m, c, t, y_test,
                                lambda m=m, c=c, t=t: ada_wsvm(m, c, t, X_train, y_train, X_test)
                            ))
                            futures.append(executor.submit(
                                safe_run_and_append,
                                f"ImADA_12_DecisionTree | {variant_tag}", testsize, m, c, t, y_test,
                                lambda m=m, t=t: imada_12_decisiontree(m, t, X_train, y_train, X_test)
                            ))
                            futures.append(executor.submit(
                                safe_run_and_append,
                                f"ImADA_12_SVM | {variant_tag}", testsize, m, c, t, y_test,
                                lambda m=m, c=c, t=t: imada_12_svm(m, c, t, X_train, y_train, X_test)
                            ))
                            futures.append(executor.submit(
                                safe_run_and_append,
                                f"ImADA_12_WSVM | {variant_tag}", testsize, m, c, t, y_test,
                                lambda m=m, c=c, t=t: imada_12_wsvm(m, c, t, X_train, y_train, X_test)
                            ))

                            # EANR-AdaBoost1: chi thay init
                            futures.append(executor.submit(
                                safe_run_and_append,
                                f"EANR-AdaBoost1_DecisionTree | {variant_tag}", testsize, m, c, t, y_test,
                                lambda m=m, t=t: eanr_adaboost1_decisiontree(m, t, X_train, y_train, X_test)
                            ))
                            futures.append(executor.submit(
                                safe_run_and_append,
                                f"EANR-AdaBoost1_SVM | {variant_tag}", testsize, m, c, t, y_test,
                                lambda m=m, c=c, t=t: eanr_adaboost1_svm(m, c, t, X_train, y_train, X_test)
                            ))
                            futures.append(executor.submit(
                                safe_run_and_append,
                                f"EANR-AdaBoost1_WSVM | {variant_tag}", testsize, m, c, t, y_test,
                                lambda m=m, c=c, t=t: eanr_adaboost1_wsvm(m, c, t, X_train, y_train, X_test)
                            ))

                            # EANR-AdaBoost2: chi thay confident
                            futures.append(executor.submit(
                                safe_run_and_append,
                                f"EANR-AdaBoost2_DecisionTree | {variant_tag}", testsize, m, c, t, y_test,
                                lambda m=m, t=t: eanr_adaboost2_decisiontree(m, t, X_train, y_train, X_test)
                            ))
                            futures.append(executor.submit(
                                safe_run_and_append,
                                f"EANR-AdaBoost2_SVM | {variant_tag}", testsize, m, c, t, y_test,
                                lambda m=m, c=c, t=t: eanr_adaboost2_svm(m, c, t, X_train, y_train, X_test)
                            ))
                            futures.append(executor.submit(
                                safe_run_and_append,
                                f"EANR-AdaBoost2_WSVM | {variant_tag}", testsize, m, c, t, y_test,
                                lambda m=m, c=c, t=t: eanr_adaboost2_wsvm(m, c, t, X_train, y_train, X_test)
                            ))

                            # EANR-AdaBoost: thay ca 2 ham
                            futures.append(executor.submit(
                                safe_run_and_append,
                                f"EANR-AdaBoost_DecisionTree | {variant_tag}", testsize, m, c, t, y_test,
                                lambda m=m, t=t: eanr_adaboost_decisiontree(m, t, X_train, y_train, X_test)
                            ))
                            futures.append(executor.submit(
                                safe_run_and_append,
                                f"EANR-AdaBoost_SVM | {variant_tag}", testsize, m, c, t, y_test,
                                lambda m=m, c=c, t=t: eanr_adaboost_svm(m, c, t, X_train, y_train, X_test)
                            ))
                            futures.append(executor.submit(
                                safe_run_and_append,
                                f"EANR-AdaBoost_WSVM | {variant_tag}", testsize, m, c, t, y_test,
                                lambda m=m, c=c, t=t: eanr_adaboost_wsvm(m, c, t, X_train, y_train, X_test)
                            ))

                # Dam bao tat ca task hoan tat.
                for future in as_completed(futures):
                    future.result()

print(f"Hoan tat. Da luu ket qua tai: {filepath}")